# Project Scenario
You are working with a customer support team. The company has a CSV file containing 80 fictional customer messages. Some messages include emails, phone numbers, URLs, order IDs, extra spaces, emojis, spelling mistakes, and different writing styles.
Your goal is to build and explore an **NLP preprocessing pipeline**, step by step. Use the **same dataset** from start to finish. This is a preprocessing assignment, not a model-training task.
## Rules
- Keep the original message and message_id columns unchanged.
- Create new columns for new results. Do not overwrite earlier results.
- Show sample outputs after each task (at least 5 rows unless stated otherwise).
- Use reusable Python functions where possible.
- Keep lists as Python lists in your notebook; you do not need to save them back to CSV.
- Never delete a character just because it looks unusual. Explain why a cleaning choice is useful for this project.
- Most processing in this task is designed for English messages.
- Important: Do POS tagging, NER, and spelling correction using readable text (message or clean_message), not heavily reduced stemmed words. These tasks are separate experiments, not always one linear chain.
## Setup
Install the libraries if needed:

    pip install pandas nltk spacy textblob
    python -m spacy download en_core_web_sm
Download needed NLTK resources for your installed version (for example, punkt, punkt_tab, stopwords, wordnet, omw-1.4, and the English POS tagger resource). Check the library documentation or the error message if a resource name differs by version.



# Part A — Data Understanding and Regex
## Task 1 — Load and Explore the CSV
- Load nlp_customer_messages.csv into a DataFrame called df.
- Print the first 5 messages, shape, and column names.
- Check missing values in each column.
- Check duplicated message_id values.
- Find the longest and shortest messages based on character count.
- Create char_count and explain what it measures.

Show: Five sample rows and a short summary of data quality.


In [22]:
import re
import pandas as pd

df = pd.read_csv("nlp_customer_messages.csv")   # change the path if your file is elsewhere

print("First 5 messages:")
print(df["message"].head(5).to_string())
print(df.shape)
print(list(df.columns))

First 5 messages:
0    Good morning, My phone is not working :(  Than...
1    Hi, Thanks for your quick response! I need hel...
2    Hi support, I LOVE this product 😍😍 Can someone...
3    Hey there, The new update is AMAZING!! Please ...
4    Dear team, The price is 250 EGP, too expensive...
(80, 2)
['message_id', 'message']


In [5]:
print(df.isna().sum())

message_id    0
message       0
dtype: int64


In [12]:
print("Duplicated message_id values:", df["message_id"].duplicated().sum())
print("Different IDs but same messages:", df["message"].duplicated().sum())

Duplicated message_id values: 0
Different IDs but same messages: 5


In [16]:
df["char_count"] = df["message"].str.len()

longest  = df.loc[df["char_count"].idxmax()]
shortest = df.loc[df["char_count"].idxmin()]

print("\nLongest message:", longest["message"])
print("\nShortest message:", shortest["message"])


Longest message: Hey there, I need help with payment for order 10612. Please check. Email: customer36@example.com Visit https://help.example.com/ticket/10612   Please   help   me.   My backup email is user.name+test@sample.org

Shortest message: Dear team, Thanks! Great service :) Thank you.


## Task 2 — Regex: Find and Extract
Use Python's re module. Create these columns:
- emails: list of email addresses in each message.
- urls: list of links in each message.
- phones: list of phone numbers in the format 010 + 8 digits.
- numbers: list of digit groups found in each message.
- has_urgent: True if the word urgent appears, ignoring case; otherwise False.

Hints: re.findall(), re.search(), re.IGNORECASE, and word boundaries.

Show: At least 5 messages containing extracted information, plus the number of messages with an email, phone, URL, and urgent keyword.


In [23]:
EMAIL_RE  = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}")
URL_RE    = re.compile(r"(?:https?://|www\.)[^\s]*[^\s.,;:!?)\"']", re.IGNORECASE)
PHONE_RE  = re.compile(r"(?<!\d)010\d{8}(?!\d)")
NUMBER_RE = re.compile(r"\d+")
URGENT_RE = re.compile(r"\burgent\b", re.IGNORECASE)

In [24]:
def extract_emails(text):  return EMAIL_RE.findall(text)
def extract_urls(text):    return URL_RE.findall(text)
def extract_phones(text):  return PHONE_RE.findall(text)
def extract_numbers(text): return NUMBER_RE.findall(text)
def has_urgent(text):      return URGENT_RE.search(text) is not None

In [26]:
df["emails"] = df["message"].apply(extract_emails)
df["urls"] = df["message"].apply(extract_urls)
df["phones"] = df["message"].apply(extract_phones)
df["numbers"] = df["message"].apply(extract_numbers)
df["has_urgent"] = df["message"].apply(has_urgent)

In [29]:
has_info = df[(df.emails.str.len() > 0) | (df.urls.str.len() > 0) | (df.phones.str.len() > 0)]
has_info[["message_id", "message", "emails", "urls", "phones", "numbers", "has_urgent"]].head(5)

,message_id,message,emails,urls,phones,numbers,has_urgent
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone...",[customer3@example.com],[],[],[3],False
3,MSG004,"Hey there, The new update is AMAZING!! Please ...",[],[https://help.example.com/ticket/10068],[],[10068],False
4,MSG005,"Dear team, The price is 250 EGP, too expensive...",[],[],[01010011705],"[250, 01010011705]",False
5,MSG006,"Dear support, The package for order 10102 is d...",[customer6@example.com],[],[],"[10102, 6]",False
7,MSG008,"Hello team, Great quality, but delivery took 7...",[],[https://help.example.com/ticket/10136],[],"[7, 10136]",False


In [30]:
print("Messages with an email  :", (df.emails.str.len() > 0).sum())
print("Messages with a phone   :", (df.phones.str.len() > 0).sum())
print("Messages with a URL     :", (df.urls.str.len() > 0).sum())
print("Messages with 'urgent'  :", df.has_urgent.sum())

Messages with an email  : 26
Messages with a phone   : 16
Messages with a URL     : 20
Messages with 'urgent'  : 11


## Task 3 — Regex: Clean Text
- Create clean_message using re.sub():
- Replace email addresses with [EMAIL].
- Replace URLs with [URL].
- Replace phone numbers with [PHONE].
- Replace remaining groups of digits with [NUMBER].
- Replace repeated whitespace with one space.
- Remove spaces from the start and end.

Important: Replace specific patterns before general numbers. Keep words, punctuation, and emojis. Keep placeholders readable.

Show: message beside clean_message for at least 8 rows.


In [33]:
def clean_text(text):
    text = EMAIL_RE.sub("[EMAIL]", text)
    text = URL_RE.sub("[URL]", text)
    text = PHONE_RE.sub("[PHONE]", text)
    text = NUMBER_RE.sub("[NUMBER]", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_message"] = df["message"].apply(clean_text)

In [36]:
df["clean_message"] = df["message"].apply(clean_text)
df[["message", "clean_message"]].head(8)

,message,clean_message
0,"Good morning, My phone is not working :( Than...","Good morning, My phone is not working :( Thank..."
1,"Hi, Thanks for your quick response! I need hel...","Hi, Thanks for your quick response! I need hel..."
2,"Hi support, I LOVE this product 😍😍 Can someone...","Hi support, I LOVE this product 😍😍 Can someone..."
3,"Hey there, The new update is AMAZING!! Please ...","Hey there, The new update is AMAZING!! Please ..."
4,"Dear team, The price is 250 EGP, too expensive...","Dear team, The price is [NUMBER] EGP, too expe..."
5,"Dear support, The package for order 10102 is d...","Dear support, The package for order [NUMBER] i..."
6,"Hello, Bad quality... very disappointed. What ...","Hello, Bad quality... very disappointed. What ..."
7,"Hello team, Great quality, but delivery took 7...","Hello team, Great quality, but delivery took [..."


## Task 4 — Test Your Regex Rules
- Create three short test strings yourself: one with an email, one with a phone number, and one with a URL.
- Run your extraction and cleaning functions on each string.
- Test a string with no email, URL, or phone.
- In 2–3 sentences, explain one limitation of your patterns.


In [39]:
email_test = "Please write to sara.ali99@mail.com for details."
phone_test = "My number is 01012345678, call me today."
url_test   = "See my screenshot at https://example.com/help/page1."
no_match   = "Thanks, the problem is solved now"

In [40]:
def run_all(text):
    print(f"--- {text!r}")
    print("   emails :", extract_emails(text))
    print("   urls   :", extract_urls(text))
    print("   phones :", extract_phones(text))
    print("   numbers:", extract_numbers(text))
    print("   urgent :", has_urgent(text))
    print("   clean  :", clean_text(text))

In [41]:
run_all(email_test)
run_all(phone_test)
run_all(url_test)
run_all(no_match)

--- 'Please write to sara.ali99@mail.com for details.'
   emails : ['sara.ali99@mail.com']
   urls   : []
   phones : []
   numbers: ['99']
   urgent : False
   clean  : Please write to [EMAIL] for details.
--- 'My number is 01012345678, call me today.'
   emails : []
   urls   : []
   phones : ['01012345678']
   numbers: ['01012345678']
   urgent : False
   clean  : My number is [PHONE], call me today.
--- 'See my screenshot at https://example.com/help/page1.'
   emails : []
   urls   : ['https://example.com/help/page1']
   phones : []
   numbers: ['1']
   urgent : False
   clean  : See my screenshot at [URL].
--- 'Thanks, the problem is solved now'
   emails : []
   urls   : []
   phones : []
   numbers: []
   urgent : False
   clean  : Thanks, the problem is solved now
